# HSE eviFluor Duo

```{device-card} hse-evifluor-duo
```

| Property | Value |
| --- | --- |
| Measurement | Fluorescence in a disposable cuvette |
| Supported excitation | 470 nm |
| Sample volume | 10 µL, following the manufacturer's workflow |
| Connection | USB virtual serial port, 115200 baud, 8N1 |
| Python import | `from pylabrobot.hse import EviFluor` |

```{warning}
This driver has not been tested on hardware. Protocol tests use a simulated serial
transport. Validate the workflow on your instrument before using experimental results,
and report hardware validation to PyLabRobot.
```

## Connection and physical setup

Connect and power the module according to its installation instructions. Attach its USB
cable to the computer. Install the serial extra with `pip install -e '.[serial]'` from
the PyLabRobot checkout. Choose the device's COM port on Windows or `/dev/ttyACM...`
port on Linux; macOS uses a `/dev/cu...` port. Omitting `port` discovers a single matching
USB VID/PID (0x1CBE/0x0003); specify a port when several modules are connected.

Mount the cuvette holder for access by your liquid handler. Prepare compatible tips,
cuvettes, the assay's low and high standards, and samples. Each empty-cuvette reading
must be paired with a filled reading of that same cuvette. The code below controls the
fluorometer; perform the indicated cuvette moves and pipetting with your liquid handler.

Communication uses PLR's serial transport and Python's standard library. No vendor C
library, DLL, or .NET runtime is needed. `use_checksum=True` enables the manufacturer's
optional CRC framing.

## Connect

Set `port` for your instrument. `setup()` checks communication by reading its firmware
version. In a script, put operations inside `try`/`finally` and `await fluor.stop()` in
the `finally` block.

In [ ]:
from pylabrobot.hse import EviFluor

fluor = EviFluor(port=None)  # Or specify e.g. "COM3" or "/dev/ttyACM0".
await fluor.setup()

## Identify the instrument

Read the identifiers without changing instrument configuration.

In [ ]:
identity = {
    "firmware": await fluor.get_firmware_version(),
    "serial_number": await fluor.get_serial_number(),
    "production_number": await fluor.get_production_number(),
}
identity

## Check the instrument

Run the firmware self-test. Zero means success; bit 0 indicates a communication fault.
Other nonzero bits are preserved for diagnosis.

In [ ]:
self_test = await fluor.run_self_test()
if self_test != 0:
    raise RuntimeError(f"eviFluor self-test failed: 0x{self_test:08x}")

## Check the holder

Start with an empty holder. Resolve any obstruction before inserting a cuvette.

In [ ]:
if not await fluor.is_cuvette_holder_empty():
    raise RuntimeError("Remove the cuvette or obstruction before starting")

## Clear the measurement history

The manufacturer's `G`/baseline command only clears the ten-reading firmware history.
It does not measure a blank or change the concentration calibration held in Python.

In [ ]:
await fluor.clear_measurements()

## Measure the high standard's empty cuvette

Pick up a tip, aspirate at least 10 µL of prepared high standard, attach a compatible
cuvette, and insert the **empty cuvette** into the holder without dispensing yet.
Measure at the device's minimum and maximum LED powers. Keep this cuvette in place.

In [ ]:
high_air_range = await fluor.measure_first_air()
high_air_range

## Dispense and measure the high standard

Dispense 10 µL of high standard into that cuvette. Autogain selects a 470 nm LED power
for the default 2000 mV target; failure raises an error. Interpolate the empty-cuvette
reading at the selected power. The high standard must be measured first because its
LED power is used for the low standard and subsequent samples.

In [ ]:
high_sample = await fluor.measure_first_sample()
power = high_sample.led_power
high_air = high_air_range.interpolate(power)
(high_air, high_sample)

## Remove the high standard cuvette

Remove and dispose of the used tip and cuvette according to your assay procedure.
Check that the holder is empty.

In [ ]:
if not await fluor.is_cuvette_holder_empty():
    raise RuntimeError("High-standard cuvette is still in the holder")

## Measure the low standard's empty cuvette

Prepare a fresh tip with at least 10 µL of low standard and a fresh empty cuvette.
Insert the empty cuvette before dispensing. Use the high standard's selected power.

In [ ]:
low_air = await fluor.measure(led_power=power)

## Dispense and measure the low standard

Dispense 10 µL of low standard into the same cuvette, then measure it at the same power.

In [ ]:
low_sample = await fluor.measure(led_power=power)

## Calculate and store the calibration

Set the actual concentrations of your prepared standards. The illustrative 0 and 10
values below must match your assay before use. All concentration results use the units
of these standards. The calculation subtracts the dark reading and the empty-cuvette
signal, then fits the line through the low and high standards.

Calibration rejects saturated readings (at least 2499 mV), unequal LED powers,
nonfinite values, and a high-standard signal that does not exceed the low standard.

In [ ]:
low_concentration = 0.0
high_concentration = 10.0  # Replace with the prepared high standard's concentration.

fluor.calibrate(
    low_air=low_air,
    low_sample=low_sample,
    high_air=high_air,
    high_sample=high_sample,
    low_concentration=low_concentration,
    high_concentration=high_concentration,
)

## Remove the low standard cuvette

Remove and dispose of the used tip and cuvette. Check the holder before preparing the sample.

In [ ]:
if not await fluor.is_cuvette_holder_empty():
    raise RuntimeError("Low-standard cuvette is still in the holder")

## Measure the sample's empty cuvette

Prepare a fresh tip with at least 10 µL of sample and a fresh empty cuvette. Insert the
empty cuvette and measure before dispensing.

In [ ]:
sample_air = await fluor.measure(led_power=power)

## Dispense and measure the sample

Dispense 10 µL of sample into that cuvette. Keep the calibrated power.

In [ ]:
sample_reading = await fluor.measure(led_power=power)
concentration = fluor.calculate_concentration(sample_air, sample_reading)
concentration

## Finish the sample

Remove and dispose of the used tip and cuvette. Apply any assay dilution factor to the
reported concentration yourself. Results outside the standards' signal range are
linearly extrapolated, so assess them against your assay's validated range.

In [ ]:
if not await fluor.is_cuvette_holder_empty():
    raise RuntimeError("Sample cuvette is still in the holder")

## Inspect measurement history

History index 0 is the newest stored reading. Reading history does not trigger another
measurement; the device holds up to ten readings.

In [ ]:
count = await fluor.get_measurement_count()
latest = await fluor.get_stored_measurement(0) if count else None
latest

## Inspect and set LED power

The LED power is an integer, not a percentage. The instrument enforces its own limits.
Changing power invalidates comparability with a calibration collected at another power.

In [ ]:
limits = await fluor.get_led_power_limits()
current_power = await fluor.get_led_power()
await fluor.set_led_power(power)
(limits, current_power)

## Select gain explicitly

For another calibration run, insert a cuvette containing its high standard before
running this cell. Always check `found`. If the power changes, collect matching
standards and update the calibration before calculating concentrations.

In [ ]:
# Run with a high-standard cuvette in place.
gain = await fluor.autogain(target=2000)
if not gain.found:
    raise RuntimeError(f"Autogain could not reach the target at power {gain.led_power}")
gain

## Set the status indicator

Available colors are `off`, `red`, `green`, `blue`, and `white`.

In [ ]:
await fluor.set_status_led("green")

## Disconnect

Close the serial port. This also clears the concentration calibration held on `fluor`.
After a timeout, corrupt frame, I/O error, or cancelled exchange, call `stop()` and
`setup()` before sending another command; late replies cannot safely be matched to
requests because the protocol has no transaction IDs.

In [ ]:
await fluor.stop()

## Protocol reference and supported scope

The implementation follows [HSE AG's evifluor source](https://github.com/hseag/evifluor/tree/f1440d49a917e2c4edb3dddd1304f5c056af0761),
used under its MIT license. The Python package includes the author's license notice.
The default request is `:<command>\n`; checksummed requests use
`;<command>@<decimal CRC>\n`. CRC-16 uses polynomial 0x1021, seed 0x1D0F, no reflection,
and no final XOR. Replies end with CR or LF and echo the command token, or return
`E <code>`. Checksums are verified when present. Quotes delimit string values.

| Operation | Request payload | Reply payload example |
| --- | --- | --- |
| Firmware / serial / production | `V 0` / `V 1` / `V 3` | `V "1.2.3"` |
| Self-test | `Y` | `Y 0` |
| Holder empty | `X` | `X 1` (1 empty, 0 occupied; optional diagnostics follow) |
| Clear history | `G` | `G` |
| History count | `V 10` | `V 2` |
| LED power / min / max | `V 15` / `V 16` / `V 17` | `V 120` |
| Set 470 nm power | `V 15 120` | `V` |
| Measure / read history | `M` / `M 0` | `M 10 2010 120 0 0 0` |
| Select gain | `C 2000` | `C 1 120` (found, LED power) |
| Indicator green | `Z 2` | `Z` |
| Error | Any command | `E 2` (invalid parameter) |

These numbers illustrate wire formatting; they are not hardware recordings. Both the
three-value measurement parsed by C# and the six-value format required by C are
accepted. The trailing triplet is preserved as `reserved_values`; its interpretation
is not assumed. C and C# disagree on registers 18–20 for the second excitation
channel, and both public measurement APIs expose 470 nm only. This driver therefore
supports 470 nm measurements and calibration. Firmware flashing, vendor JSON run
storage, and 625 nm register control are outside this API.

A nonzero self-test result needs investigation. `EviFluorError` retains the firmware
error code and command, including unknown codes. Invalid frames raise
`EviFluorProtocolError`; incomplete replies raise `TimeoutError`.